# LeetCode 765 — Couples Holding Hands bằng Python

- Link: https://leetcode.com/problems/couples-holding-hands/
- Độ khó: **Hard**
- Chủ đề: Tham lam (Greedy), Union-Find (DSU), Đồ thị & đếm thành phần liên thông / chu trình, BFS

Skeleton trên LeetCode (Python3):

```python
class Solution:
    def minSwapsCouples(self, row: List[int]) -> int:
```

## 1. Giới thiệu & mục tiêu học

Có `n` cặp đôi ngồi trên `2n` ghế xếp hàng. Mỗi lần ta được **đổi chỗ hai người bất kỳ**. Hỏi cần **ít nhất** bao nhiêu lần đổi để mọi cặp đôi ngồi cạnh nhau (trên cùng một "ghế đôi" `(0,1)`, `(2,3)`, ...)?

Bài này rất hay vì có **lời giải tham lam cực ngắn**, nhưng để hiểu **vì sao tham lam tối ưu** ta cần nhìn bài toán như **đồ thị** và đếm **thành phần liên thông / chu trình** — một kỹ thuật xuất hiện trong mọi bài "số lần hoán đổi ít nhất để sắp xếp".

Sau notebook này bạn sẽ:

1. Viết lời giải BFS vét cạn (baseline) cho `n` nhỏ.
2. Hiểu và cài đặt tham lam "sửa từng ghế đôi từ trái sang phải" với bảng vị trí `pos`.
3. Mô hình hoá thành đồ thị: **ghế đôi = đỉnh**, **cặp đôi = cạnh**, và chứng minh `đáp án = n − số thành phần liên thông`.
4. Cài đặt **Union-Find** (có nén đường đi) và **DFS đếm chu trình**.
5. Nhận ra mối liên hệ với bài "số lần hoán đổi tối thiểu để sắp hoán vị" (`n − số chu trình`).

## 2. Đề bài & ví dụ

**Phát biểu lại.** Có `2n` người, đánh số `0 .. 2n−1`, ngồi trên hàng ghế; `row[i]` là người ngồi ghế `i`. Các cặp đôi là `(0, 1), (2, 3), ..., (2n−2, 2n−1)`. Một **lần đổi**: chọn hai người bất kỳ, cho họ đổi ghế. Trả về số lần đổi **ít nhất** để mọi cặp đôi ngồi cạnh nhau, tức là với mọi `k`, hai ghế `2k` và `2k+1` chứa một cặp đôi.

**Ví dụ 1**

```text
Input: row = [0,2,1,3]
Output: 1
Giải thích: đổi chỗ người 2 (ghế 1) và người 1 (ghế 2) -> [0,1,2,3].
```

**Ví dụ 2**

```text
Input: row = [3,2,0,1]
Output: 0
Giải thích: cặp (2,3) và (0,1) đã ngồi cạnh nhau.
```

**Ràng buộc**

- `2n == row.length`
- `2 <= n <= 30`
- `0 <= row[i] < 2n`
- Mọi phần tử của `row` **khác nhau**.

Mẹo nhỏ: bạn đời của người `x` là `x ^ 1` (XOR với 1): `0↔1`, `2↔3`, `4↔5`...

In [1]:
from typing import List

EXAMPLES = [([0, 2, 1, 3], 1), ([3, 2, 0, 1], 0)]

def all_seated(row):
    return all(row[i] ^ 1 == row[i + 1] for i in range(0, len(row), 2))

print([x ^ 1 for x in range(6)])
print(all_seated([0, 2, 1, 3]), all_seated([3, 2, 0, 1]))

[1, 0, 3, 2, 5, 4]
False True


## 3. Kiến thức nền

### 3.1. Phép XOR để tìm bạn đời, phép chia nguyên để tìm "mã cặp"

- `x ^ 1`: lật bit cuối → bạn đời.
- `x // 2` (hoặc `x >> 1`): **mã cặp đôi** — hai người cùng cặp có cùng `x // 2`.
- `i // 2`: **mã ghế đôi** của ghế `i`.

In [2]:
for x in range(6):
    print(f"người {x}: bạn đời {x ^ 1}, mã cặp {x // 2}")
print("ghế 0..5 thuộc ghế đôi:", [i // 2 for i in range(6)])

người 0: bạn đời 1, mã cặp 0
người 1: bạn đời 0, mã cặp 0
người 2: bạn đời 3, mã cặp 1
người 3: bạn đời 2, mã cặp 1
người 4: bạn đời 5, mã cặp 2
người 5: bạn đời 4, mã cặp 2
ghế 0..5 thuộc ghế đôi: [0, 0, 1, 1, 2, 2]


### 3.2. Bảng vị trí (inverse permutation)

`pos[x]` = ghế mà người `x` đang ngồi. Khi đổi chỗ hai người, phải cập nhật **cả** `row` lẫn `pos` để giữ đồng bộ.

In [3]:
row = [0, 2, 1, 3]
pos = [0] * len(row)
for i, x in enumerate(row):
    pos[x] = i
print("pos =", pos)

def swap_seats(row, pos, i, j):
    row[i], row[j] = row[j], row[i]
    pos[row[i]], pos[row[j]] = i, j

swap_seats(row, pos, 1, 2)
print(row, pos)

pos = [0, 2, 1, 3]
[0, 1, 2, 3] [0, 1, 2, 3]


### 3.3. Union-Find (Disjoint Set Union)

Quản lý các tập rời nhau với hai thao tác: `find(x)` (tìm đại diện) và `union(a, b)` (gộp hai tập). Dùng **nén đường đi** để gần như `O(1)` mỗi thao tác. Đếm số tập còn lại bằng biến `count`.

In [4]:
class DSU:
    def __init__(self, n):
        self.parent = list(range(n))
        self.count = n

    def find(self, x):
        while self.parent[x] != x:
            self.parent[x] = self.parent[self.parent[x]]   # nén đường đi (halving)
            x = self.parent[x]
        return x

    def union(self, a, b):
        ra, rb = self.find(a), self.find(b)
        if ra == rb:
            return False
        self.parent[ra] = rb
        self.count -= 1
        return True

d = DSU(5)
d.union(0, 1); d.union(3, 4); d.union(1, 0)
print("số tập:", d.count, [d.find(i) for i in range(5)])

số tập: 3 [1, 1, 2, 4, 4]


### 3.4. Số lần hoán đổi tối thiểu để sắp một hoán vị = `n − số chu trình`

Hoán vị `p` gồm các **chu trình**. Một chu trình độ dài `L` cần đúng `L − 1` lần đổi chỗ để "gỡ" (mỗi lần đổi đặt đúng một phần tử, và lần cuối đặt đúng hai). Tổng: `Σ (L − 1) = n − (số chu trình)`. Bài này dùng cùng tư tưởng, nhưng trên **ghế đôi** và **cặp đôi**.

In [5]:
def cycles_of_perm(p):
    seen, cyc = [False] * len(p), 0
    for i in range(len(p)):
        if not seen[i]:
            cyc += 1
            j = i
            while not seen[j]:
                seen[j] = True
                j = p[j]
    return cyc

p = [1, 2, 0, 4, 3, 5]      # chu trình (0 1 2), (3 4), (5)
print("chu trình:", cycles_of_perm(p), "-> số lần đổi tối thiểu:", len(p) - cycles_of_perm(p))

chu trình: 3 -> số lần đổi tối thiểu: 3


## 4. Ý tưởng nền tảng / trực giác

### 4.1. Tính tay ví dụ 1: `[0, 2, 1, 3]`

- Ghế đôi 0 (ghế 0, 1): `0, 2`. Người `0` cần bạn `1` — đang ở ghế 2. Đổi người ghế 1 (`2`) với người ghế 2 (`1`) → `[0, 1, 2, 3]`.
- Ghế đôi 1 (ghế 2, 3): `2, 3` — đã đúng.

Tổng 1 lần.

### 4.2. Một ví dụ "vòng": `row = [0, 3, 2, 5, 4, 1]`

Ghế đôi: `{0,3}`, `{2,5}`, `{4,1}`. Ghi theo **mã cặp** (`x // 2`): `{0,1}`, `{1,2}`, `{2,0}`. Ba ghế đôi và ba cặp tạo thành một **vòng**: ghế đôi A chứa nửa cặp 0 và nửa cặp 1, ghế đôi B chứa nửa cặp 1 và nửa cặp 2, ghế đôi C chứa nửa cặp 2 và nửa cặp 0. Vòng có 3 ghế đôi → cần `3 − 1 = 2` lần đổi:

- Ghế đôi 0: `0, 3` → bạn của 0 là 1 (ghế 5). Đổi ghế 1 ↔ ghế 5: `[0, 1, 2, 5, 4, 3]`.
- Ghế đôi 1: `2, 5` → bạn của 2 là 3 (ghế 5). Đổi ghế 3 ↔ ghế 5: `[0, 1, 2, 3, 4, 5]`.

### 4.3. Trực giác đồ thị

Mỗi ghế đôi là một **đỉnh**. Mỗi cặp đôi có hai người, ngồi ở hai ghế đôi (có thể trùng) → nối hai ghế đôi đó bằng một **cạnh**. Mỗi đỉnh có đúng **bậc 2** (hai người trên ghế đôi) → đồ thị là hợp của các **chu trình** rời nhau. Một chu trình gồm `L` ghế đôi cần `L − 1` lần đổi. Đáp án = `Σ (L − 1) = n − (số chu trình)`.

In [6]:
row = [0, 3, 2, 5, 4, 1]
print("theo mã cặp:", [[row[i] // 2, row[i + 1] // 2] for i in range(0, len(row), 2)])

theo mã cặp: [[0, 1], [1, 2], [2, 0]]


## 5. Cách 1 — Brute Force: BFS trên mọi cách sắp xếp

**Ý tưởng.** Mỗi trạng thái là một bộ `row`. Từ một trạng thái, thử **mọi** cặp ghế `(i, j)` để đổi. BFS cho số bước ít nhất tới trạng thái mà mọi cặp đôi ngồi cạnh nhau. Chỉ khả thi với `2n ≤ 8`.

In [7]:
from collections import deque

def brute(row):
    start = tuple(row)
    if all_seated(start):
        return 0
    m = len(row)
    dist = {start: 0}
    q = deque([start])
    while q:
        u = q.popleft()
        for i in range(m):
            for j in range(i + 1, m):
                v = list(u)
                v[i], v[j] = v[j], v[i]
                v = tuple(v)
                if v not in dist:
                    dist[v] = dist[u] + 1
                    if all_seated(v):
                        return dist[v]
                    q.append(v)
    return -1

for row, exp in EXAMPLES:
    print(row, "->", brute(row), "(kỳ vọng", exp, ")")
print(brute([0, 3, 2, 5, 4, 1]))

[0, 2, 1, 3] -> 1 (kỳ vọng 1 )
[3, 2, 0, 1] -> 0 (kỳ vọng 0 )
2


**Độ phức tạp.** Số trạng thái tới `(2n)!`, mỗi trạng thái có `C(2n, 2)` láng giềng → `O((2n)! · n^2 · n)`. Với `2n = 60` là không tưởng.

**Nút thắt.** BFS không biết đổi chỗ nào là "có ích". Câu hỏi: có phải lúc nào cũng nên đổi để **ghép xong ít nhất một cặp** không? Và đổi theo thứ tự nào?

## 6. Bước đột phá

**Insight 1 — Mỗi lần đổi ghép xong được ít nhất một cặp, tốt nhất là đi từ trái sang.** Xét ghế đôi `(i, i+1)` với `i` chẵn. Nếu `row[i+1]` không phải bạn của `row[i]`, đổi `row[i+1]` với bạn của `row[i]` (dù đang ở đâu). Sau đó ghế đôi này **cố định mãi mãi**, ta không bao giờ chạm lại.

**Insight 2 — Đồ thị ghế đôi – cặp đôi.** Như mục 4.3: đồ thị `n` đỉnh, `n` cạnh, mọi đỉnh bậc 2 → các chu trình rời nhau. Gọi `C` là số chu trình (một ghế đôi đã đúng là chu trình độ dài 1 — một khuyên).

- **Cận dưới:** một lần đổi chỗ chỉ thay đổi hai ghế đôi, nên làm số chu trình tăng **tối đa 1**. Trạng thái đích có `n` chu trình (toàn khuyên). Vậy cần ít nhất `n − C` lần đổi.
- **Đạt được:** mỗi bước tham lam tách đúng một đỉnh khỏi chu trình của nó thành khuyên → số chu trình tăng đúng 1 (khi chu trình còn ≥ 2 đỉnh).

⇒ **Đáp án = `n − C`**, và tham lam đạt đúng giá trị này.

**Quy tắc nếu… thì…**

- Nếu `row[i+1] == row[i] ^ 1` → ghế đôi đã xong, bỏ qua.
- Ngược lại → tìm `pos[row[i] ^ 1]`, đổi với ghế `i+1`, **cập nhật `pos`**, tăng đáp án.
- Nếu chỉ cần **số** lần đổi (không cần thao tác) → đếm thành phần liên thông bằng Union-Find: `n − components`.

## 7. Các cách tiếp theo

### 7.1. Cách 2 — Tham lam, tìm bạn đời bằng quét tuyến tính

**Giải thích.** Bản đơn giản nhất: với mỗi ghế đôi, nếu chưa đúng thì **quét** phần còn lại của hàng để tìm bạn đời. `O(n^2)` — hoàn toàn đủ với `n ≤ 30`.

In [8]:
def greedy_scan(row):
    row = row[:]
    swaps = 0
    for i in range(0, len(row), 2):
        partner = row[i] ^ 1
        if row[i + 1] == partner:
            continue
        j = row.index(partner, i + 2)        # bạn đời nằm sau ghế đôi i
        row[i + 1], row[j] = row[j], row[i + 1]
        swaps += 1
    return swaps

for row, exp in EXAMPLES:
    print(greedy_scan(row), exp)

1 1
0 0


**Code nộp LeetCode (Cách 2).**

In [9]:
class SolutionScan:
    def minSwapsCouples(self, row: List[int]) -> int:
        swaps = 0
        for i in range(0, len(row), 2):
            partner = row[i] ^ 1
            if row[i + 1] != partner:
                j = row.index(partner, i + 2)
                row[i + 1], row[j] = row[j], row[i + 1]
                swaps += 1
        return swaps

print([SolutionScan().minSwapsCouples(r[:]) for r, _ in EXAMPLES])

[1, 0]


**Độ phức tạp.** `n` ghế đôi × quét `O(n)` → `O(n^2)`. Bộ nhớ `O(1)` thêm (sửa tại chỗ).

### 7.2. Cách 3 — Tham lam với bảng vị trí `pos` (O(n))

**Giải thích.** Thay `row.index` bằng tra `pos[partner]` trong `O(1)`. Mỗi lần đổi chỗ cập nhật `pos` cho hai người bị di chuyển.

In [10]:
def greedy_pos(row):
    row = row[:]
    pos = [0] * len(row)
    for i, x in enumerate(row):
        pos[x] = i
    swaps = 0
    for i in range(0, len(row), 2):
        partner = row[i] ^ 1
        if row[i + 1] == partner:
            continue
        j = pos[partner]
        # đổi người ở ghế i+1 với partner ở ghế j
        other = row[i + 1]
        row[i + 1], row[j] = partner, other
        pos[partner], pos[other] = i + 1, j
        swaps += 1
    return swaps

for row, exp in EXAMPLES:
    print(greedy_pos(row), exp)

1 1
0 0


**Code nộp LeetCode (Cách 3).**

In [11]:
class SolutionPos:
    def minSwapsCouples(self, row: List[int]) -> int:
        pos = [0] * len(row)
        for i, x in enumerate(row):
            pos[x] = i
        swaps = 0
        for i in range(0, len(row), 2):
            partner = row[i] ^ 1
            if row[i + 1] != partner:
                j = pos[partner]
                other = row[i + 1]
                row[i + 1], row[j] = partner, other
                pos[partner], pos[other] = i + 1, j
                swaps += 1
        return swaps

print([SolutionPos().minSwapsCouples(r[:]) for r, _ in EXAMPLES])

[1, 0]


**Độ phức tạp.** `O(n)` thời gian, `O(n)` bộ nhớ cho `pos`.

### 7.3. Cách 4 — Union-Find trên mã cặp đôi

**Giải thích.** Không cần mô phỏng đổi chỗ. Với mỗi ghế đôi chứa hai người `a, b`, **gộp** hai cặp đôi `a // 2` và `b // 2`. Sau khi xử lý mọi ghế đôi, mỗi thành phần liên thông (theo cặp đôi) tương ứng một chu trình ở mục 6. Đáp án = `n − số thành phần`.

(Ở đây ta lấy **cặp đôi** làm đỉnh, **ghế đôi** làm cạnh — đồ thị "đối ngẫu" với mục 6, nhưng cũng là các chu trình với cùng số đỉnh/cạnh, nên kết quả như nhau.)

In [12]:
def union_find(row):
    n = len(row) // 2
    dsu = DSU(n)
    for i in range(0, len(row), 2):
        dsu.union(row[i] // 2, row[i + 1] // 2)
    return n - dsu.count

for row, exp in EXAMPLES:
    print(union_find(row), exp)

1 1
0 0


**Code nộp LeetCode (Cách 4).**

In [13]:
class SolutionDSU:
    def minSwapsCouples(self, row: List[int]) -> int:
        n = len(row) // 2
        parent = list(range(n))

        def find(x):
            while parent[x] != x:
                parent[x] = parent[parent[x]]
                x = parent[x]
            return x

        swaps = 0
        for i in range(0, len(row), 2):
            a, b = find(row[i] // 2), find(row[i + 1] // 2)
            if a != b:
                parent[a] = b
                swaps += 1          # mỗi lần gộp thành công = giảm 1 thành phần
        return swaps

print([SolutionDSU().minSwapsCouples(r[:]) for r, _ in EXAMPLES])

[1, 0]


Mẹo: `n − components` = **số lần `union` thành công** (mỗi lần gộp giảm số thành phần đi 1). Nên ta chỉ cần đếm số lần gộp.

**Độ phức tạp.** `O(n · α(n))` ≈ `O(n)`. Bộ nhớ `O(n)`.

### 7.4. Cách 5 — Dựng đồ thị tường minh và đếm chu trình bằng DFS

**Giải thích.** Đỉnh = ghế đôi `0..n−1`. Với mỗi cặp đôi `c`, hai người `2c` và `2c+1` ngồi ở ghế đôi `pos[2c] // 2` và `pos[2c+1] // 2` → thêm cạnh giữa hai đỉnh đó (có thể là khuyên). Đếm số thành phần liên thông bằng DFS lặp; đáp án `n − C`.

In [14]:
def cycles_dfs(row):
    n = len(row) // 2
    pos = [0] * len(row)
    for i, x in enumerate(row):
        pos[x] = i
    adj = [[] for _ in range(n)]
    for c in range(n):
        u, v = pos[2 * c] // 2, pos[2 * c + 1] // 2
        adj[u].append(v)
        adj[v].append(u)
    seen = [False] * n
    comps = 0
    for s in range(n):
        if seen[s]:
            continue
        comps += 1
        stack = [s]
        seen[s] = True
        while stack:
            u = stack.pop()
            for v in adj[u]:
                if not seen[v]:
                    seen[v] = True
                    stack.append(v)
    return n - comps

for row, exp in EXAMPLES:
    print(cycles_dfs(row), exp)

1 1
0 0


**Code nộp LeetCode (Cách 5)** — bản gọn: vì mọi đỉnh bậc 2, ta có thể **đi dọc chu trình** mà không cần danh sách kề. Ta "đến" một ghế đôi qua một người `came`; người **còn lại** `leave` trên ghế đôi đó dẫn ta sang ghế đôi chứa bạn đời `leave ^ 1`, và cứ thế cho đến khi quay về ghế đã thăm.

In [15]:
class SolutionCycle:
    def minSwapsCouples(self, row: List[int]) -> int:
        n = len(row) // 2
        pos = [0] * len(row)
        for i, x in enumerate(row):
            pos[x] = i
        seen = [False] * n
        cycles = 0
        for s in range(n):
            if seen[s]:
                continue
            cycles += 1
            k, came = s, row[2 * s]               # coi như đến ghế đôi s qua người bên trái
            while not seen[k]:
                seen[k] = True
                a, b = row[2 * k], row[2 * k + 1]
                leave = b if a == came else a     # rời đi qua người CÒN LẠI
                mate = leave ^ 1
                k, came = pos[mate] // 2, mate    # sang ghế đôi chứa bạn đời
        return n - cycles

print([SolutionCycle().minSwapsCouples(r[:]) for r, _ in EXAMPLES])

[1, 0]


Điểm tinh tế: phải rời ghế đôi qua người **còn lại**, không phải luôn qua người bên phải (xem Lỗi 4 ở mục 13).

**Độ phức tạp.** `O(n)` thời gian và bộ nhớ.

## 8. TRACE từng bước

Trace **Cách 3** trên `row = [5, 4, 2, 6, 3, 1, 0, 7]` (n = 4).

| ghế đôi | row trước | partner của row[i] | row[i+1] | hành động | swaps |
|---|---|---|---|---|---|
| 0 (ghế 0,1) | `5 4 2 6 3 1 0 7` | 4 | 4 | đúng rồi | 0 |
| 1 (ghế 2,3) | `5 4 2 6 3 1 0 7` | 3 (ghế 4) | 6 | đổi ghế 3 ↔ 4 | 1 |
| 2 (ghế 4,5) | `5 4 2 3 6 1 0 7` | 7 (ghế 7) | 1 | đổi ghế 5 ↔ 7 | 2 |
| 3 (ghế 6,7) | `5 4 2 3 6 7 0 1` | 1 | 1 | đúng rồi | 2 |

In [16]:
def trace_greedy(row):
    row = row[:]
    pos = [0] * len(row)
    for i, x in enumerate(row):
        pos[x] = i
    swaps = 0
    for i in range(0, len(row), 2):
        partner = row[i] ^ 1
        before = row[:]
        if row[i + 1] == partner:
            print(f"ghế đôi {i // 2}: {before}  {row[i]} & {row[i + 1]} đã là cặp")
            continue
        j = pos[partner]
        other = row[i + 1]
        row[i + 1], row[j] = partner, other
        pos[partner], pos[other] = i + 1, j
        swaps += 1
        print(f"ghế đôi {i // 2}: {before}  bạn của {row[i]} là {partner} ở ghế {j} -> đổi ghế {i + 1}<->{j}: {row}  swaps={swaps}")
    return swaps

trace_greedy([5, 4, 2, 6, 3, 1, 0, 7])

ghế đôi 0: [5, 4, 2, 6, 3, 1, 0, 7]  5 & 4 đã là cặp
ghế đôi 1: [5, 4, 2, 6, 3, 1, 0, 7]  bạn của 2 là 3 ở ghế 4 -> đổi ghế 3<->4: [5, 4, 2, 3, 6, 1, 0, 7]  swaps=1
ghế đôi 2: [5, 4, 2, 3, 6, 1, 0, 7]  bạn của 6 là 7 ở ghế 7 -> đổi ghế 5<->7: [5, 4, 2, 3, 6, 7, 0, 1]  swaps=2
ghế đôi 3: [5, 4, 2, 3, 6, 7, 0, 1]  0 & 1 đã là cặp


2

Trace **Cách 4 (Union-Find)** trên cùng input: in các cặp đôi được gộp ở mỗi ghế đôi.

In [17]:
def trace_dsu(row):
    n = len(row) // 2
    dsu = DSU(n)
    for i in range(0, len(row), 2):
        a, b = row[i] // 2, row[i + 1] // 2
        merged = dsu.union(a, b)
        groups = {}
        for c in range(n):
            groups.setdefault(dsu.find(c), []).append(c)
        print(f"ghế đôi {i // 2}: người {row[i]},{row[i + 1]} -> cặp {a},{b}  "
              f"{'gộp' if merged else 'cùng nhóm/khuyên'}  nhóm={list(groups.values())}")
    print("đáp án = n - số nhóm =", n, "-", dsu.count, "=", n - dsu.count)

trace_dsu([5, 4, 2, 6, 3, 1, 0, 7])

ghế đôi 0: người 5,4 -> cặp 2,2  cùng nhóm/khuyên  nhóm=[[0], [1], [2], [3]]
ghế đôi 1: người 2,6 -> cặp 1,3  gộp  nhóm=[[0], [1, 3], [2]]
ghế đôi 2: người 3,1 -> cặp 1,0  gộp  nhóm=[[0, 1, 3], [2]]
ghế đôi 3: người 0,7 -> cặp 0,3  cùng nhóm/khuyên  nhóm=[[0, 1, 3], [2]]
đáp án = n - số nhóm = 4 - 2 = 2


## 9. Code nộp LeetCode

Bản chọn: **Cách 3 (tham lam + pos)** — `O(n)`, dễ giải thích; kèm phiên bản Union-Find ở trên làm phương án thay thế.

In [18]:
class Solution:
    def minSwapsCouples(self, row: List[int]) -> int:
        pos = [0] * len(row)                 # pos[x] = ghế của người x
        for i, x in enumerate(row):
            pos[x] = i
        swaps = 0
        for i in range(0, len(row), 2):      # xét từng ghế đôi (i, i+1)
            partner = row[i] ^ 1             # bạn đời của người ngồi ghế i
            if row[i + 1] != partner:
                j = pos[partner]
                other = row[i + 1]
                row[i + 1], row[j] = partner, other    # kéo partner về ghế i+1
                pos[partner], pos[other] = i + 1, j    # đồng bộ bảng vị trí
                swaps += 1
        return swaps

for row, exp in EXAMPLES:
    assert Solution().minSwapsCouples(row[:]) == exp
print("OK")

OK


Giải thích từng dòng:

- `pos`: đảo ngược của `row`, cho phép tìm ghế của bạn đời trong `O(1)`.
- `range(0, len(row), 2)`: chỉ xét ghế chẵn — mỗi ghế đôi một lần.
- `row[i] ^ 1`: bạn đời.
- Ta **giữ nguyên** người ở ghế `i` và kéo bạn đời về ghế `i+1` (giữ người ghế `i+1` lại cũng được — đối xứng).
- `j > i + 1` luôn đúng: các ghế đôi trước đã "đóng", nên bạn đời chưa ghép phải ở phía sau.

## 10. Vì sao thuật toán đúng

**Mô hình.** Đồ thị `G`: đỉnh = ghế đôi; với mỗi cặp đôi, nối hai ghế đôi chứa hai người của cặp. `n` đỉnh, `n` cạnh, mọi đỉnh bậc 2 → `G` là hợp rời các chu trình (khuyên = chu trình độ dài 1, cạnh kép = chu trình độ dài 2). Gọi `C` = số chu trình. Trạng thái đích ⇔ `C = n`.

**Cận dưới.** Một lần đổi chỗ hai người chỉ thay đổi các cạnh gắn với **hai** ghế đôi chứa họ. Phân tích các trường hợp: số chu trình thay đổi `−1`, `0` hoặc `+1` — không bao giờ tăng quá 1 (đổi chỗ trong cùng chu trình có thể tách nó làm 2; đổi giữa hai chu trình khác nhau gộp chúng làm 1). Từ `C` đến `n` cần **ít nhất `n − C`** lần đổi.

**Tham lam đạt cận dưới.** Khi ghế đôi `k` chưa đúng, nó nằm trên chu trình độ dài `L ≥ 2`. Đổi người `row[2k+1]` với bạn đời của `row[2k]` biến ghế đôi `k` thành một khuyên, phần còn lại thành chu trình độ dài `L − 1` → `C` tăng đúng 1. Mỗi bước tham lam đều là bước "tốt nhất có thể" → tổng số bước = `n − C`.

**Union-Find / DFS** đếm trực tiếp `C` (số thành phần liên thông = số chu trình vì mỗi thành phần là một chu trình). ∎

## 11. Phân tích Time/Space

| Cách | Time | Space |
|---|---|---|
| 1. BFS | `O((2n)! · n^3)` | `O((2n)! · n)` |
| 2. Tham lam + quét | `O(n^2)` | `O(1)` |
| 3. Tham lam + pos | `O(n)` | `O(n)` |
| 4. Union-Find | `O(n α(n))` | `O(n)` |
| 5. DFS đếm chu trình | `O(n)` | `O(n)` |

In [19]:
import random, time
random.seed(765)
rows = []
for _ in range(3000):
    r = list(range(60)); random.shuffle(r); rows.append(r)
for name, f in [("scan", greedy_scan), ("pos", greedy_pos), ("dsu", union_find), ("dfs", cycles_dfs)]:
    t0 = time.perf_counter()
    for r in rows:
        f(r)
    print(f"{name:5} {1000 * (time.perf_counter() - t0):7.1f} ms cho 3000 hàng 60 ghế")

scan     29.1 ms cho 3000 hàng 60 ghế


pos      21.3 ms cho 3000 hàng 60 ghế


dsu      30.0 ms cho 3000 hàng 60 ghế


dfs      29.6 ms cho 3000 hàng 60 ghế


## 12. Bộ test

In [20]:
methods = {
    "scan": greedy_scan,
    "pos": greedy_pos,
    "dsu": union_find,
    "dfs": cycles_dfs,
    "SolutionScan": lambda r: SolutionScan().minSwapsCouples(r[:]),
    "SolutionPos": lambda r: SolutionPos().minSwapsCouples(r[:]),
    "SolutionDSU": lambda r: SolutionDSU().minSwapsCouples(r[:]),
    "SolutionCycle": lambda r: SolutionCycle().minSwapsCouples(r[:]),
    "Solution": lambda r: Solution().minSwapsCouples(r[:]),
}

# 12.1 Ví dụ đề
for name, f in methods.items():
    for row, exp in EXAMPLES:
        assert f(row) == exp, (name, row)

# 12.2 Edge cases
edge = [
    ([0, 1], 0),
    ([1, 0], 0),                      # cặp ngồi ngược thứ tự vẫn hợp lệ
    ([0, 2, 1, 3], 1),
    ([0, 3, 2, 5, 4, 1], 2),          # một vòng 3 ghế đôi
    ([0, 2, 4, 6, 7, 1, 3, 5], 3),
    (list(range(60)), 0),
]
for name, f in methods.items():
    for row, exp in edge:
        assert f(row) == exp, (name, row, f(row))
print("Ví dụ + edge cases: OK")

Ví dụ + edge cases: OK


In [21]:
# 12.3 Đối chiếu BFS trên MỌI hoán vị 6 ghế và ngẫu nhiên 8 ghế
from itertools import permutations
for p in permutations(range(6)):
    b = brute(list(p))
    for name, f in methods.items():
        assert f(list(p)) == b, (name, p)
random.seed(1)
for _ in range(150):
    r = list(range(8)); random.shuffle(r)
    b = brute(r)
    for name, f in methods.items():
        assert f(r) == b, (name, r)
print("Khớp BFS: OK")

# 12.4 Ngẫu nhiên lớn: các cách nhanh khớp nhau
for _ in range(2000):
    m = 2 * random.randint(1, 30)
    r = list(range(m)); random.shuffle(r)
    assert len({f(r) for f in methods.values()}) == 1, r
print("Ngẫu nhiên lớn: OK")

Khớp BFS: OK


Ngẫu nhiên lớn: OK


## 13. Lỗi người mới hay gặp

### Lỗi 1 — Quên cập nhật `pos` sau khi đổi chỗ

**Code sai:**

In [22]:
def wrong_no_pos_update(row):
    row = row[:]
    pos = {x: i for i, x in enumerate(row)}
    swaps = 0
    for i in range(0, len(row), 2):
        partner = row[i] ^ 1
        if row[i + 1] != partner:
            j = pos[partner]
            row[i + 1], row[j] = row[j], row[i + 1]     # SAI: không cập nhật pos
            swaps += 1
    return swaps

r = [5, 4, 2, 6, 3, 1, 0, 7]
print("sai:", wrong_no_pos_update(r), " đúng:", greedy_pos(r), " hàng sau khi 'sửa' có đúng không?")
def final_row_wrong(row):
    row = row[:]; pos = {x: i for i, x in enumerate(row)}
    for i in range(0, len(row), 2):
        p = row[i] ^ 1
        if row[i + 1] != p:
            j = pos[p]; row[i + 1], row[j] = row[j], row[i + 1]
    return row, all_seated(row)
print(final_row_wrong([0, 2, 4, 6, 7, 1, 3, 5]))

sai: 2  đúng: 2  hàng sau khi 'sửa' có đúng không?
([0, 6, 4, 2, 7, 5, 3, 1], False)


**Vì sao sai:** người `other` bị chuyển tới ghế `j` nhưng `pos[other]` vẫn ghi ghế cũ. Lần sau tra `pos[other]` sẽ đổi nhầm chỗ — kết quả cuối cùng có thể không hợp lệ dù số đếm "trông có vẻ đúng".

**Code đúng:** `pos[partner], pos[other] = i + 1, j`.

### Lỗi 2 — Kiểm tra cặp đôi bằng `abs(a - b) == 1`

**Code sai:**

In [23]:
def wrong_abs(row):
    return all(abs(row[i] - row[i + 1]) == 1 for i in range(0, len(row), 2))

a, b = 1, 2
print("abs(a-b)==1:", abs(a - b) == 1, "| a^1==b:", a ^ 1 == b)
print("đếm số ghế đôi 'đúng' trên [1,2,0,3,5,4]:",
      sum(abs(x - y) == 1 for x, y in [(1, 2), (0, 3), (5, 4)]), "(sai) vs",
      sum(x ^ 1 == y for x, y in [(1, 2), (0, 3), (5, 4)]), "(đúng)")

abs(a-b)==1: True | a^1==b: False
đếm số ghế đôi 'đúng' trên [1,2,0,3,5,4]: 2 (sai) vs 1 (đúng)


**Vì sao sai:** `1` và `2` hơn kém 1 nhưng **không** phải cặp đôi (cặp là `(0,1)`, `(2,3)`).

**Code đúng:** `row[i] ^ 1 == row[i+1]` hoặc `row[i] // 2 == row[i+1] // 2`.

### Lỗi 3 — Đếm "số ghế đôi sai" làm đáp án

**Code sai:**

In [24]:
def wrong_count_bad(row):
    return sum(1 for i in range(0, len(row), 2) if row[i] ^ 1 != row[i + 1])   # SAI

r = [0, 2, 1, 3]
print("sai:", wrong_count_bad(r), " đúng:", greedy_pos(r))

sai: 2  đúng: 1


**Vì sao sai:** một lần đổi có thể sửa **hai** ghế đôi cùng lúc (chu trình độ dài 2). Đáp án là `Σ (L − 1)` chứ không phải `Σ L`.

**Code đúng:** `n − số chu trình`.

### Lỗi 4 — "Đi dọc chu trình" luôn theo người bên phải

**Code sai:**

In [25]:
def wrong_cycle_walk(row):
    n = len(row) // 2
    pos = [0] * len(row)
    for i, x in enumerate(row):
        pos[x] = i
    seen = [False] * n
    cycles = 0
    for s in range(n):
        if seen[s]:
            continue
        cycles += 1
        k = s
        while not seen[k]:
            seen[k] = True
            right = row[2 * k + 1]            # SAI: luôn đi theo người bên phải
            k = pos[right ^ 1] // 2
    return n - cycles

random.seed(4)
bad = None
for _ in range(3000):
    m = 2 * random.randint(1, 5)
    r = list(range(m)); random.shuffle(r)
    if wrong_cycle_walk(r) != greedy_pos(r) and (bad is None or len(r) < len(bad)):
        bad = r
print("phản ví dụ:", bad, "sai =", wrong_cycle_walk(bad), "đúng =", greedy_pos(bad))

phản ví dụ: [0, 5, 3, 4, 1, 2] sai = 1 đúng = 2


**Vì sao sai:** khi nhảy tới ghế đôi chứa `right ^ 1`, nếu `right ^ 1` lại ngồi **bên phải** ghế đó thì "người bên phải" chính là `right ^ 1`, và bạn đời của nó (`right`) dẫn ta **quay về** ghế đôi vừa rời — vòng `while` dừng dù chưa đi hết chu trình. Phần còn lại của chu trình bị đếm thành một "chu trình" khác → số chu trình bị đếm dư → đáp án nhỏ hơn thực tế.

**Code đúng:** rời ghế đôi qua người **còn lại** (không phải người vừa dẫn tới), như `SolutionCycle` ở mục 7.4.

In [26]:
for _ in range(3000):
    m = 2 * random.randint(1, 30)
    r = list(range(m)); random.shuffle(r)
    assert SolutionCycle().minSwapsCouples(r[:]) == greedy_pos(r)
print("SolutionCycle (đi qua người còn lại): OK")

SolutionCycle (đi qua người còn lại): OK


Bài học: khi "đi dọc" một chu trình trong đồ thị bậc 2, luôn rời đỉnh qua **cạnh còn lại**, không phải qua một phía cố định.

## 14. Mẹo & nhận dạng pattern

- **"Số lần đổi chỗ ít nhất để sắp xếp"** → đếm chu trình: đáp án `n − số chu trình` (*2471. Minimum Number of Operations to Sort a Binary Tree by Level*, *Minimum Swaps to Sort*).
- **Mọi đỉnh bậc 2** → thành phần liên thông = chu trình → Union-Find hoặc DFS.
- **Bạn đời / cặp `(2k, 2k+1)`** → `x ^ 1`, `x // 2`.
- **Tham lam "sửa từ trái sang, không bao giờ chạm lại"** thường tối ưu khi mỗi bước tăng một đại lượng tiềm năng (ở đây: số chu trình) đúng 1.
- **Union-Find**: `n − components` = số lần `union` thành công.

## 15. Biến thể

1. **Chỉ được đổi chỗ hai người ngồi cạnh nhau** → bài khác hẳn (đếm nghịch thế / BFS), tham lam này không còn đúng.
2. **Ghế xếp vòng tròn** (ghế `2n−1` cạnh ghế `0`) và cặp có thể ngồi ở `(2k+1, 2k+2)` → phải thử hai cách chia ghế đôi, lấy min.
3. **Nhóm 3 người (bộ ba)** → đồ thị bậc 3, không còn là chu trình; bài trở nên khó hơn nhiều.
4. **In ra dãy thao tác đổi chỗ** → dùng Cách 3 và ghi lại `(i+1, j)`.
5. **Số lần đổi tối thiểu để sắp một hoán vị** → `n − cycles(p)`.

In [27]:
# Biến thể 4: in thao tác
def swap_ops(row):
    row = row[:]
    pos = [0] * len(row)
    for i, x in enumerate(row):
        pos[x] = i
    ops = []
    for i in range(0, len(row), 2):
        partner = row[i] ^ 1
        if row[i + 1] != partner:
            j = pos[partner]; other = row[i + 1]
            row[i + 1], row[j] = partner, other
            pos[partner], pos[other] = i + 1, j
            ops.append((i + 1, j))
    return ops, row

print(swap_ops([5, 4, 2, 6, 3, 1, 0, 7]))

# Biến thể 2: hàng ghế vòng tròn
def circular_min_swaps(row):
    shifted = row[1:] + row[:1]          # ghế đôi (1,2), (3,4), ..., (2n-1, 0)
    return min(greedy_pos(row), greedy_pos(shifted))

r = [1, 2, 3, 4, 5, 0]       # ghế 5 (người 0) và ghế 0 (người 1) kề nhau theo vòng
print("chia ghế thường:", greedy_pos(r), " cho phép chia lệch theo vòng:", circular_min_swaps(r))

([(3, 4), (5, 7)], [5, 4, 2, 3, 6, 7, 0, 1])
chia ghế thường: 2  cho phép chia lệch theo vòng: 0


## 16. Bảng so sánh các cách

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. BFS | Thử mọi phép đổi | Giai thừa | Giai thừa | Kiểm thử `2n ≤ 8` |
| 2. Tham lam + quét | Sửa từng ghế đôi, `row.index` | `O(n^2)` | `O(1)` | Code ngắn nhất, đủ với `n ≤ 30` |
| 3. Tham lam + pos | Như 2, tra vị trí `O(1)` | `O(n)` | `O(n)` | Nộp bài, cần in thao tác |
| 4. Union-Find | Gộp mã cặp trên mỗi ghế đôi | `O(n α)` | `O(n)` | Chỉ cần con số, chứng minh rõ |
| 5. DFS chu trình | Dựng đồ thị ghế đôi, đếm thành phần | `O(n)` | `O(n)` | Hiểu bản chất đồ thị |

## 17. Bài tập

**Bài 1 (đoán output).** `minSwapsCouples([0, 4, 7, 3, 1, 5, 2, 8, 6, 9])` = ?

**Bài 2 (tính tay).** Với `row = [2, 0, 5, 4, 3, 1]`, vẽ đồ thị ghế đôi – cặp đôi, đếm chu trình, suy ra đáp án.

**Bài 3 (điền TODO).** Hoàn thiện `min_swaps_to_sort(p)` — số lần đổi chỗ tối thiểu để sắp hoán vị `p` của `0..n-1`.

**Bài 4 (tự viết lại).** Viết lại Union-Find có **hợp theo kích thước** (union by size).

**Bài 5 (mini challenge).** Viết hàm trả về **dãy thao tác** tối thiểu và kiểm chứng: áp dụng lần lượt thì mọi cặp ngồi cạnh nhau, và độ dài dãy bằng đáp án BFS.

In [28]:
# Bài 3 — TODO (không gọi khi chạy notebook)
def min_swaps_to_sort(p):
    # TODO: đếm chu trình của p, trả len(p) - cycles
    pass

# Bài 4 — TODO
class DSUSize:
    def __init__(self, n):
        # TODO: parent, size, count
        pass
    def find(self, x):
        pass
    def union(self, a, b):
        # TODO: gắn gốc nhỏ vào gốc lớn
        pass

# Bài 5 — TODO
def min_swap_sequence(row):
    # TODO: tham lam + pos, ghi lại (i+1, j)
    pass

## 18. Gợi ý 3 mức

**Bài 1.**
- Mức 1: ghế đôi theo mã cặp: `{0,2}, {3,1}, {0,2}, {1,4}, {3,4}`.
- Mức 2: nối các mã cặp: 0–2, 1–3, 0–2, 1–4, 3–4 → thành phần `{0,2}`, `{1,3,4}`.
- Mức 3: `n = 5`, 2 thành phần → đáp án `3`.

**Bài 2.**
- Mức 1: mã cặp từng ghế đôi: `{1,0}, {2,2}, {1,0}`.
- Mức 2: ghế đôi 1 là khuyên; ghế đôi 0 và 2 tạo chu trình độ dài 2.
- Mức 3: `C = 2`, `n = 3` → `1`.

**Bài 3.**
- Mức 1: đánh dấu `seen`.
- Mức 2: từ `i` nhảy `j = p[j]` đến khi gặp lại.
- Mức 3: `len(p) - cycles`.

**Bài 4.**
- Mức 1: mảng `size` khởi tạo 1.
- Mức 2: so sánh `size[ra]` và `size[rb]`.
- Mức 3: gắn gốc nhỏ vào gốc lớn, cộng size.

**Bài 5.**
- Mức 1: tái sử dụng Cách 3.
- Mức 2: áp dụng từng `(a, b)` lên bản sao `row`.
- Mức 3: assert `all_seated` và `len(ops) == brute(row)`.

## 19. Đáp án tham khảo

In [29]:
# Bài 1
assert Solution().minSwapsCouples([0, 4, 7, 3, 1, 5, 2, 8, 6, 9]) == 3 == union_find([0, 4, 7, 3, 1, 5, 2, 8, 6, 9])

# Bài 2
assert Solution().minSwapsCouples([2, 0, 5, 4, 3, 1]) == 1 == brute([2, 0, 5, 4, 3, 1])

# Bài 3
def min_swaps_to_sort_ans(p):
    return len(p) - cycles_of_perm(p)

def brute_sort_swaps(p):
    start, goal = tuple(p), tuple(sorted(p))
    dist, q = {start: 0}, deque([start])
    while q:
        u = q.popleft()
        if u == goal:
            return dist[u]
        for i in range(len(u)):
            for j in range(i + 1, len(u)):
                v = list(u); v[i], v[j] = v[j], v[i]; v = tuple(v)
                if v not in dist:
                    dist[v] = dist[u] + 1; q.append(v)

for p in permutations(range(5)):
    assert min_swaps_to_sort_ans(list(p)) == brute_sort_swaps(list(p))

# Bài 4
class DSUSizeAns:
    def __init__(self, n):
        self.parent = list(range(n)); self.size = [1] * n; self.count = n
    def find(self, x):
        while self.parent[x] != x:
            self.parent[x] = self.parent[self.parent[x]]; x = self.parent[x]
        return x
    def union(self, a, b):
        ra, rb = self.find(a), self.find(b)
        if ra == rb:
            return False
        if self.size[ra] < self.size[rb]:
            ra, rb = rb, ra
        self.parent[rb] = ra; self.size[ra] += self.size[rb]; self.count -= 1
        return True

def union_find_size(row):
    n = len(row) // 2
    d = DSUSizeAns(n)
    for i in range(0, len(row), 2):
        d.union(row[i] // 2, row[i + 1] // 2)
    return n - d.count

# Bài 5
def min_swap_sequence_ans(row):
    return swap_ops(row)[0]

random.seed(77)
for _ in range(300):
    r = list(range(8)); random.shuffle(r)
    ops = min_swap_sequence_ans(r)
    cur = r[:]
    for a, b in ops:
        cur[a], cur[b] = cur[b], cur[a]
    assert all_seated(cur) and len(ops) == brute(r)
    assert union_find_size(r) == len(ops)
print("Tất cả đáp án đúng")

Tất cả đáp án đúng


## 20. Template nhớ nhanh + Checklist trước khi submit

**Template tham lam:**

```python
pos = [0] * len(row)
for i, x in enumerate(row): pos[x] = i
swaps = 0
for i in range(0, len(row), 2):
    p = row[i] ^ 1
    if row[i+1] != p:
        j, other = pos[p], row[i+1]
        row[i+1], row[j] = p, other
        pos[p], pos[other] = i+1, j
        swaps += 1
return swaps
```

**Template Union-Find:** với mỗi ghế đôi, `union(row[i]//2, row[i+1]//2)`; đáp án = số lần union thành công.

**Checklist trước khi submit:**

- [ ] Bạn đời là `x ^ 1` (không phải `x + 1` hay `abs(a-b)==1`).
- [ ] Chỉ duyệt ghế chẵn `range(0, len(row), 2)`.
- [ ] Cập nhật **cả** `row` và `pos` khi đổi chỗ.
- [ ] Union-Find: dùng **mã cặp** `x // 2`, không phải số người.
- [ ] Đáp án = `n − số thành phần` (không phải số ghế đôi sai).
- [ ] Tên method đúng: `minSwapsCouples(self, row)`.